# Gemini API

> Translate Gemini generateContent requests, responses, and streams to and from fastllm's shared message types, including tools, thought signatures, search grounding, and media

In [1]:
#| default_exp gemini

## Imports

In [2]:
#| export
import html
from collections import Counter
from fastcore.utils import *
from fastcore.meta import *
from fasttransport.errors import api_error_from_event

from fastllm.types import *
from aidialog.msg_parts import (MdStr, Part, PartType, Msg, Completion, Text, Thinking, ToolUse, ToolResult, tool_text,
    InputImage, InputAudio, InputVideo, InputFile, sys_text, part_txt, data_url, url_mime)
from fastllm.streaming import *
from fastllm.streaming import mk_acollect_stream

In [3]:
#| hide
import yaml,base64,httpx2
from fasttransport.core import AsyncHttpCli
from fastcore.test import *
from fastcore.funccall import get_schema
from cachy.core import enable_cachy, disable_cachy, doms
from aidialog.msg_parts import fmt2hist, hist2fmt
from fastllm.acomplete import acomplete, defaults

In [4]:
enable_cachy(doms=doms+sample_doms, hdrs=('content-type',))


In [5]:
#| hide
defaults.reasoning_effort = 'l'

Gemini puts the model and method in the request URL. `endpoint` selects `generateContent` for an ordinary response or `streamGenerateContent` with `alt=sse` for a stream. The examples use `gem_call` to post through `provider_req`, just as `acomplete` does.

In [6]:
#| export
def endpoint(model, stream):
    "Gemini routes the model and method in the URL; streams add SSE"
    return f'v1beta/{model}:{"streamGenerateContent" if stream else "generateContent"}', ({'alt': 'sse'} if stream else None)

In [7]:
gem_cli = AsyncHttpCli('https://generativelanguage.googleapis.com', base_headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})
async def gem_call(model, stream=False, **body):
    path,params = endpoint(model, stream)
    return await provider_req(gem_cli, path, body, params=params, stream=stream)

## Normalize

These helpers read the first response candidate and build shared tool calls, usage, and message parts. `mk_completion` combines them into a `Completion`.

### Tool Calls

Function calls carry a name, arguments, and optional ID. `_fn_calls` retains the other fields on each part in `ToolUse.raw`, including any thought signature. Without a provider ID it assigns `call_<part index>`.

In [8]:
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

In [9]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [10]:
sch = lite_mk_func(simple_add)
sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

Keep thought signatures on their original parts when replaying a response. Gemini 3 validates the first function-call part of each step in the current turn. A missing signature causes a 400 error, even with Gemini 3 Flash's `minimal` thinking setting. The validation doesn't extend to earlier turns. See Google's [generateContent signature rules](https://ai.google.dev/gemini-api/docs/generate-content/thought-signatures).

The first example below copies the extra part fields from `ToolUse.raw` when constructing the tool round trip. Later, `Msg.raw` lets us replay the whole model message.

In [11]:
#| export
def _url_cite(c): return dict(type='url_citation', url=c['web']['uri'], title=c['web'].get('title', ''))

def grounding(cand):
    "The `url_citation`s and the `google_search` call a candidate's `groundingMetadata` or `toolCall` parts describe, or `([], None)` without either; the call's `text` lists the sources and the Google Search suggestion links"
    gm = cand.get('groundingMetadata') or {}
    tcs = [p['toolCall'] for p in nested_idx(cand, 'content', 'parts') or [] if 'toolCall' in p]
    if not gm and not tcs: return [], None
    cites = [_url_cite(c) for c in gm.get('groundingChunks', []) if nested_idx(c, 'web', 'uri')]
    chips = re.findall(r'<a class="chip" href="([^"]+)">([^<]*)</a>', nested_idx(gm, 'searchEntryPoint', 'renderedContent') or '')
    queries = gm.get('webSearchQueries') or [q for tc in tcs for q in (tc.get('args') or {}).get('queries', [])]
    lines = [f"{c['title']}: {c['url']}" for c in cites] + [f"Search suggestion: [{html.unescape(t)}]({html.unescape(u)})" for u,t in chips]
    call = ToolUse(id='google_search', name='google_search', arguments=dict(queries=queries), server=True, text=MdStr('\n'.join(lines)))
    return cites, call

`norm_tool_calls` puts the synthesized search call first, followed by the candidate's `functionCall` parts.

In [12]:
#| export
def _fn_call(p, i):
    fc = p['functionCall']
    return ToolUse(id=fc.get("id", f"call_{i}"), name=fc.get("name", ""), arguments=fc.get("args") or {},
        raw={k:v for k,v in p.items() if k != "functionCall"})

def _fn_calls(parts): return [_fn_call(p, i) for i,p in enumerate(parts) if p.get('functionCall')]

def norm_tool_calls(resp):
    "Tool calls of a response: the `google_search` call its grounding describes, then its `functionCall` parts"
    cand = nested_idx(resp, 'candidates', 0) or {}
    _,call = grounding(cand)
    return ([call] if call else []) + _fn_calls(nested_idx(cand, 'content', 'parts') or [])

User defined:

In [13]:
mn = flash

In [14]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "What is 3 + 5 and 10+ 20? Use the simple_add tool in parallel."}]}], tools=[{"functionDeclarations": [sch['function']]}])
norm_tool_calls(resp)

[ToolUse(raw={'thoughtSignature': 'Ep8DCpwDAWkUfRMaFc1X51bAmigZm+4QeefLsAXClm5cGDSQESkuPwYWwhDpLYZJZbVkopbVykzMuE4O+dQkgFGpWvw8P8DcfEPhfQw3AWPm+Vg0yq8sy6pEMEO7u56EVFdGljnZR/47A1kLpCbFgqfTho0YqIj2MVCA6gxXWMKf9Q8L15MOYSYPj/SftfN98GIaQYvNLgncasib5Gy7iuXC7Qr6Qva7/g6ijt5wMr7JIRNJTaEXoNrU59QripJyq+LSqbwLs7aUU63ClMdUFGYvE78gPIORi0s6TyULTo66D7fII20gb8k3BVF6rYtiw1L7t3qt52uWRlhJ7twjCR3d700jwlIS3CL4aUbMEQzh9yJOz3vYWE1JLMs30wt+XZl/eGyAY3k5pykm02oWNisuQZUYiCUG+cA9WohOYKsON+IHwQwWpmLyeYLSWhXy0mHqsk0/ElNC4T0YnVW8MYYDAA/v2Ht+PC0EQbncIvYxg2xc5pw45NE6S3LRLwZQqkNd+OOSnr6Bv6xR/z1VOybF5gnIu22pHKkVGV/MxILsUQ=='}, cache_control=None, id='call_151675', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, text=None),
 ToolUse(raw={}, cache_control=None, id='call_151676', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, text=None)]

In [15]:
model_parts = [{"functionCall": {"name": tc.name, "args": tc.arguments}, **tc.raw} for tc in  norm_tool_calls(resp)]
func_results = [{'functionResponse': {'name': 'simple_add', 'response': {'result': 8}}},
    {'functionResponse': {'name': 'simple_add', 'response': {'result': 30}}}]
resp_gem = await gem_call(
    model=mn,
    contents=[
        {"role": "user",  "parts": [{"text": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]},
        {"role": "model", "parts": model_parts},
        {"role": "user",  "parts": func_results},
    ],
    tools=[{"functionDeclarations": [sch['function']]}])
resp_gem

```python
{ 'candidates': [{'content': {'parts': [{'text': '* 3 + 5 = 8\n* 10 + 20 = 30', 'thoughtSignature': 'EmkKZwFpFH0Te9Y+qX0/OPLPzx/xgM9Yd/x1YPeoK9ZBT7JoZo6hGlQQjdz9FFf1Y7Pc+pOzwm0Q7Z3TqvGfxwpMvqmdFDHyrfEDtX/o86Pwimfz7LKZarexQ7ZlHjMywzTMgrMnh/ZvFTg='}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}],
  'modelVersion': 'gemini-3.8-flash',
  'responseId': 'k_m6ao7WE_ef1e8PzdiE8Ac',
  'usageMetadata': { 'candidatesTokenCount': 22,
                     'promptTokenCount': 247,
                     'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 247}],
                     'serviceTier': 'standard',
                     'totalTokenCount': 269}}
```

The search example reports sources in `groundingMetadata`. `grounding` converts them to URL citations and creates one server-side `google_search` call. Its arguments contain the search queries. Its Markdown result lists the sources and suggestion links from the `searchEntryPoint` widget.

When metadata has no queries, the helper looks for them in `toolCall` parts. With neither metadata nor tool-call parts, it returns no search call.


In [16]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}], tools=[{"googleSearch": {}}])
norm_tool_calls(resp)

[ToolUse(raw=None, cache_control=None, id='google_search', name='google_search', arguments={'queries': ['Istanbul weather current conditions celsius', 'weather in Istanbul today']}, server=True, text='timeanddate.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEyiGtMLfk9sHK6R94-W5VGB8Kr3PDPyaBelSDYeQ_-mmLjftnEBNTVckpsZBwytflTfP-rJGOZ4sk_ZjF-yn3WTDS8tOsGQYh0nDIbmhGhW35KZkgh55gHTc1gt-GqsJLDy7seXp6j9zkJWnXw\ntheweathernetwork.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFLYm7IrFahIcrc5cQ7A1NvJOQLpfwulwl5rlcS1MNOMUWJLKPzuyk7KpJe_mvcILxaBSrX6fXSqF6dGjz8D5nWNyoHb9me8-uh0zWL5pTr5Xv50xUZ4i8A7wP9uWDXdKO-uJpRKE8VWVbS_xKd4_lVieQEbIPVrahv1kjk\naccuweather.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH4G0x6aLvd6u1WLsHvj00goVRqpZauZqobGvxBH9X7LI1jlS2Vet56YrYij8fN-K0e_5wYPFcyzB0xECqYq8m-hI8DKGav2Fqa1TycQ6T0vMhrj1MYs7f3VGN21qIdRil9bUZfbMPAgzj-z1ytCKm5JWPAnfqNGaDYGMdtezc=\nyandex.com.tr: https://vertexaisearch.cloud.goo

In [17]:
str(nested_idx(resp, 'candidates', 0, 'groundingMetadata'))[:100]

"{'searchEntryPoint': {'renderedContent': '<style>\\n.container {\\n  align-items: center;\\n  border-ra"

Code execution:

In [18]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using the code execution tool"}]}], tools=[{"codeExecution": {}}])
norm_tool_calls(resp)

[]

In [19]:
resp

```python
{ 'candidates': [{'content': {'parts': [{'executableCode': {'language': 'PYTHON', 'code': 'def fibonacci(n):\n    fib_sequence = []\n    a, b = 0, 1\n    for _ in range(n):\n        fib_sequence.append(a)\n        a, b = b, a + b\n    return fib_sequence\n\nfirst_10_fib = fibonacci(10)\nprint(first_10_fib)', 'id': 'call_144971'}, 'thoughtSignature': 'EscECsQEAWkUfRPNQIPMAGFzeCWVHVN8WCvcO7kJjgj0bhWJQyPqJ3GdEdigHbNahcWo9aJLbTe9hptiuRK1RExhdNlDqrqytQQGXhzstUYWnluXn9S8fSBiUPJiLP6+d+NETdicb8P0iks43oGRaZp1EXhmVM40nCoBHzeWaxwOwVj/5UI5s+w+ovc0QcCeDuzjta+dGRPl9lSKi7gT3Bo9+QKML9ZLyTuksLhzoKtap7f08YNrGVF5CD8EhV2f4CclkUDGKXLFg1gLMXqnHx5gK6fmSEXvJOigQFM/VNjYoqwTzPnpwsyI75kWTBSoxwTO1f1vbKd13wUr/i1UDjD/n4AbJUjLywaFedebKaApuMWFFKfzVy6tpJojzy9PcFDDHLhixVZoqAgihis/UGUMv35ewI7/2nX4ESu/gniW+CA74FYex1nVK9esufzoe1ojB7XamiLaz0TLmztIfhWAxftuq1PENSGiXQGSKoYEvkLLO4gZEPtgU7/zwS6rwH60GqYkhiyAopo3yJIPNVK8Go+knKc1xFR9D0n7Fj7aKnT6jJRDk8jcUjZzL/a6fQhmiPi6QZ32sejftWGlWuSPeo3eBS2b6hbItBcwJeVzzIf1d/36JIXBauq0exqcK8/0RbJZ03cFUUiaEa37TxveCQjfF8CiAa+MQAYkurflA6svPjpJPRt/PWt2ex+1meV1ml2GRkYHGFjwVDhfPhHxWzTWtj1C21Lj4jWtm5IZLshN5khztqz61Ja433FYaeS5PtEw76+0gQ=='}, {'codeExecutionResult': {'outcome': 'OUTCOME_OK', 'output': '[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]\n', 'id': 'call_144971'}}, {'text': 'The first 10 Fibonacci numbers (starting with $F_0 = 0$) are:\n\n**0, 1, 1, 2, 3, 5, 8, 13, 21, 34**\n\n*(Note: If starting with $F_1 = 1$, the first 10 numbers are **1, 1, 2, 3, 5, 8, 13, 21, 34, 55**).*', 'thoughtSignature': 'EmkKZwFpFH0Tmxlk4Dft+a7zTNLJX3RQ6PFAYg5q3emAqJsATXGuC/gmjMB2J30wHvQB0IT4S/o5Vr7zD47NC2VNuIcqsZadTmInHb4CCVH/OMrugD93juAp255OxE8L4x5EXaNNyy1RZc4='}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}],
  'modelVersion': 'gemini-3.8-flash',
  'responseId': 'mvm6av7QGa202roPt5rK8QY',
  'usageMetadata': { 'candidatesTokenCount': 208,
                     'promptTokenCount': 235,
                     'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 235}],
                     'serviceTier': 'standard',
                     'thoughtsTokenCount': 220,
                     'toolUsePromptTokenCount': 364,
                     'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 364}],
                     'totalTokenCount': 1027}}
```

In [20]:
str(nested_idx(resp, 'candidates', 0, 'content', 'parts', 0))[:100]

"{'executableCode': {'language': 'PYTHON', 'code': 'def fibonacci(n):\\n    fib_sequence = []\\n    a, "

In [21]:
str(nested_idx(resp, 'candidates', 0, 'content', 'parts', 1))

"{'codeExecutionResult': {'outcome': 'OUTCOME_OK', 'output': '[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]\\n', 'id': 'call_144971'}}"

The computer-use example returns a client function call. The caller must perform the requested action and send a result; this notebook only inspects the call.

In [22]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "Go to google.com and search for 'hello world'"}]}], tools=[{"computerUse": {"environment": "ENVIRONMENT_BROWSER"}}])
norm_tool_calls(resp)

[ToolUse(raw={'thoughtSignature': 'EuwBCukBAWkUfROMxextHAd6FHxABwYst2ht/CJmVlhkwyKib3WZhKHkC/qDdxWh/Tjzt1ejWP/ySL1Mt3LXPkuLcbhKV39JquvNw+TqblkRgKh+D/iuSNm0z4gNwDYtTyZz5vJYP3yPQVrtOWJdShnDlo/hhFCS/uohPyh3IzUSfyCQDtdm7ZpgCRhwHbLnuE6t1VwB77a0t39K++GdG9AmUN6inKIoK7tJsH9fBsenOgBh4FBwz23Vgu5ANznfc2invEUfl/rPZ0Be82vVwZM14cGA8dMVXRckPrAB3oTVKVC2XHqrz/wyCG/rBTI='}, cache_control=None, id='call_165658', name='navigate', arguments={'url': 'https://www.google.com', 'intent': 'Navigate to google.com'}, server=False, text=None)]

### Usage

`norm_usage` reads input, candidate-output, total, cached, and thinking token counts from `usageMetadata`. It adds server-tool counts to the raw usage dict. Each `executableCode` part counts as one execution. Search counts the reported queries, or one request if a search has no query list.

In [23]:
#| export
def norm_usage(resp):
    "Normalize Gemini usageMetadata shape."
    if not (usg:=resp.get("usageMetadata")): return None
    pt = int(usg.get("promptTokenCount", 0) or 0)
    ct = int(usg.get("candidatesTokenCount", 0) or 0)
    tt = int(usg.get("totalTokenCount", pt + ct) or (pt + ct))
    cached = int(usg.get("cachedContentTokenCount", 0) or 0)
    reasoning = int(usg.get("thoughtsTokenCount", 0) or 0)
    parts = nested_idx(resp, 'candidates', 0, 'content', 'parts') or []
    cand = nested_idx(resp, 'candidates', 0) or {}
    stu = {}
    if any("executableCode" in p for p in parts): stu["code_execution"] = sum(1 for p in parts if "executableCode" in p)
    if call := grounding(cand)[1]: stu['google_search'] = len(call.arguments['queries'] or [1])
    if stu: usg['server_tool_use'] = stu
    return Usage(prompt_tokens=pt, completion_tokens=ct, total_tokens=tt,
        cached_tokens=cached, reasoning_tokens=reasoning, search_queries=stu.get('google_search', 0), raw=usg)

Text response:

In [24]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "hi!"}]}])
norm_usage(resp)

Usage(prompt_tokens=3, completion_tokens=9, total_tokens=168, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=156, search_queries=0, raw={'promptTokenCount': 3, 'candidatesTokenCount': 9, 'totalTokenCount': 168, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3}], 'thoughtsTokenCount': 156, 'serviceTier': 'standard'})

Server tools:

In [25]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}], tools=[{"googleSearch": {}}])
norm_usage(resp)

Usage(prompt_tokens=134, completion_tokens=317, total_tokens=727, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=276, search_queries=2, raw={'promptTokenCount': 134, 'candidatesTokenCount': 317, 'totalTokenCount': 727, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 134}], 'thoughtsTokenCount': 276, 'serviceTier': 'standard', 'server_tool_use': {'google_search': 2}})

In [26]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code"}]}], tools=[{"codeExecution": {}}])
norm_usage(resp)

Usage(prompt_tokens=49, completion_tokens=320, total_tokens=577, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=37, search_queries=0, raw={'promptTokenCount': 49, 'candidatesTokenCount': 320, 'totalTokenCount': 577, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 49}], 'toolUsePromptTokenCount': 171, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 171}], 'thoughtsTokenCount': 37, 'serviceTier': 'standard', 'server_tool_use': {'code_execution': 1}})

In [27]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "What is solveit? https://solve.it.com/"}]}], tools=[{"urlContext": {}}])
norm_usage(resp)

Usage(prompt_tokens=46, completion_tokens=470, total_tokens=4025, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=328, search_queries=1, raw={'promptTokenCount': 46, 'candidatesTokenCount': 470, 'totalTokenCount': 4025, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 46}], 'toolUsePromptTokenCount': 3181, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3181}], 'thoughtsTokenCount': 328, 'serviceTier': 'standard', 'server_tool_use': {'google_search': 1}})

In [28]:
str(nested_idx(resp, 'candidates', 0, 'content', 'parts', 0, 'text'))[:300]

'**Solveit** (from [solve.it.com](https://solve.it.com/)) is an educational course and software platform created by **Jeremy Howard** (founder of fast.ai) and **Eric Ries** (author of *The Lean Startup*) through **Answer.AI**. \n\nIt focuses on using AI effectively for programming, writing, and problem'

### Finish Reason

The converter lowercases Gemini's reason and maps `max_tokens` to `length`, and `safety` or `blocklist` to `content_filter`. A stopped response with a client tool call becomes `tool_calls`. Other reasons pass through.

In [29]:
#| export
def norm_finish(resp, tcs=None):
    "Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter."
    reason = nested_idx(resp, 'candidates', 0, 'finishReason')
    if reason is not None: reason = reason.lower()
    mp = dict(stop=FinishReason.stop, max_tokens=FinishReason.length, safety=FinishReason.content_filter, blocklist=FinishReason.content_filter)
    r =  mp.get(reason, reason)
    return FinishReason.tool_calls if r==FinishReason.stop and any(~L(tcs).attrgot('server')) else r 

In [30]:
norm_finish(resp)

<finish_reason.stop: 'stop'>

### Non-stream Completion

`norm_parts` keeps text and thinking as separate parts and puts grounding citations on the first `Text`. Code-execution blocks become `ToolResult` parts with their original data in `raw`. Search `toolCall` and `toolResponse` blocks give way to the synthesized search call. Function-call parts become `ToolUse` parts in their original order.

In [31]:
#| export
def _gem_part_type(p):
    "Map Gemini part to canonical PartType."
    if 'functionCall' in p: return PartType.tool_use
    if 'executableCode' in p or 'codeExecutionResult' in p: return PartType.tool_result
    if p.get('thought'): return PartType.thinking
    return PartType.text

def norm_parts(resp):
    "Normalize Gemini generateContent response; a search's `toolCall` and `toolResponse` parts give way to the call `grounding` builds"
    c0 = nested_idx(resp, 'candidates', 0) or {}
    cites,call = grounding(c0)
    parts = [call] if call else []
    for i,p in enumerate(nested_idx(c0, 'content', 'parts') or []):
        if 'toolCall' in p or 'toolResponse' in p: continue
        typ = _gem_part_type(p)
        if typ == PartType.tool_use:      parts.append(_fn_call(p, i))
        elif typ == PartType.thinking:    parts.append(Thinking(p.get("text",""), raw=p))
        elif typ == PartType.tool_result: parts.append(ToolResult(text=p.get("text",""), raw=p))
        else:                             parts.append(Text(p.get("text",""), raw=p))
    if txts := [p for p in parts if isinstance(p, Text)]: txts[0].citations = cites
    return parts

`raw_msg` returns the first candidate's model content without rebuilding it. `Msg.raw` keeps those parts, including signatures, for replay.

In [32]:
#| export
def raw_msg(resp):
    "The model content as the API sent it"
    return nested_idx(resp, 'candidates', 0, 'content')

Register the normalizers so `mk_completion` can find them by provider name. `raw_msg` supplies the assistant message's replay data.

In [33]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, raw_msg=raw_msg)
api_registry.register('gemini', **norm_funcs)

#### Text

In [34]:
api_name,vnd_nm = 'gemini', 'gemini'

In [35]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "Say hi in one word"}]}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

Hello

::: details

- model: `models/gemini-3.8-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=6, completion_tokens=1, total_tokens=138, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=131, search_queries=0, raw={'promptTokenCount': 6, 'candidatesTokenCount': 1, 'totalTokenCount': 138, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 131, 'serviceTier': 'standard'})`

:::

#### Thinking parts

In this small input to the normalizer, `thought=True` distinguishes the thought summary from the answer. The streaming examples below use a recorded model response.

In [36]:
comp = mk_completion({"candidates": [{"content": {"parts": [{"text": "Let me think...", "thought": True}, {"text": "Here's the answer."}], "role": "model"}, "finishReason": "STOP"}], "usageMetadata": dict(promptTokenCount=10, candidatesTokenCount=20, totalTokenCount=30)}, mn, api_name, vnd_nm)
comp


::: details
## Thinking

Let me think...

:::

Here's the answer.

::: details

- model: `models/gemini-3.8-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=10, completion_tokens=20, total_tokens=30, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'promptTokenCount': 10, 'candidatesTokenCount': 20, 'totalTokenCount': 30})`

:::

#### Tool call

In [37]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "What is 3 + 5? Use simple_add tool."}]}], tools=[{"functionDeclarations": [sch['function']]}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp



🔧 simple_add({'b': 5, 'a': 3})


::: details

- model: `models/gemini-3.8-flash`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=66, completion_tokens=18, total_tokens=136, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=52, search_queries=0, raw={'promptTokenCount': 66, 'candidatesTokenCount': 18, 'totalTokenCount': 136, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 66}], 'thoughtsTokenCount': 52, 'serviceTier': 'standard'})`

:::

In [38]:
test_eq(comp.tool_calls[0].server, False)

Web search response:

In [39]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}], tools=[{"googleSearch": {}}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp



🔧 google_search({'queries': ['Istanbul weather current conditions celsius', 'weather in Istanbul today']})
The weather in Istanbul is cool, breezy, and mostly cloudy with intermittent light rain and showers:

* **Temperature:** Around **17°C** (62°F–63°F), with daytime highs reaching about **18°C – 20°C** (64°F – 68°F) and dropping to around **15°C – 16°C** (59°F – 61°F) overnight.
* **Conditions:** Mostly cloudy / overcast with scattered light rain and drizzle.
* **Wind:** Breezy northerly winds (around 25–30 km/h / 15–18 mph, with stronger gusts), making it feel noticeably cooler than the actual temperature. 
* **Humidity:** High, around 80%–87%.

If you are heading out, a light water-resistant jacket or umbrella is recommended.

::: details

- model: `models/gemini-3.8-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=134, completion_tokens=317, total_tokens=727, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=276, search_queries=2, raw={'promptTokenCount': 134, 'candidatesTokenCount': 317, 'totalTokenCount': 727, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 134}], 'thoughtsTokenCount': 276, 'serviceTier': 'standard', 'server_tool_use': {'google_search': 2}})`

:::

In [40]:
test_eq(comp.usage.search_queries, len(comp.tool_calls[0].arguments['queries']))


Code Execution:

In [41]:
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code execution tool"}]}], tools=[{"codeExecution": {}}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

The first 10 Fibonacci numbers (starting from $F_0 = 0$) are:

**0, 1, 1, 2, 3, 5, 8, 13, 21, 34**

*(Note: If starting the sequence from $F_1 = 1$, the first 10 numbers are: **1, 1, 2, 3, 5, 8, 13, 21, 34, 55**)*

::: details

- model: `models/gemini-3.8-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=243, completion_tokens=212, total_tokens=1057, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=229, search_queries=0, raw={'promptTokenCount': 243, 'candidatesTokenCount': 212, 'totalTokenCount': 1057, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 243}], 'toolUsePromptTokenCount': 373, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 373}], 'thoughtsTokenCount': 229, 'serviceTier': 'standard', 'server_tool_use': {'code_execution': 1}})`

:::

In [42]:
test_eq(comp.usage.raw['server_tool_use']['code_execution'], 1)

### Streaming

`norm_sse_event` turns each response chunk into a `Delta`. The shared collector uses `PartAccum` to assemble the final `Completion` while yielding text, thinking, and citation updates.

In [43]:
print_stream = PrintStream()

This converter reads complete `functionCall` arguments from each chunk. It doesn't collect JSON argument fragments. A provider that streams fragments would instead put them in `ToolUse.arguments` under `_delta`:

```py
ToolUse(id=tc.get("id",""), name=fn.get("name",""), arguments={'_delta': fn.get("arguments")}, raw=raw)
```

That fragment convention also differs from Responses' `response.output_item.done` event. Here we join each chunk's text by its `thought` flag and read its finish reason and usage. Search deltas wait for `groundingMetadata`; an error payload raises an API error.

In [44]:
#| export
def norm_sse_event(ev, **kwargs):
    "Normalize Gemini stream event into Delta."
    ev = obj2dict(ev)
    cand = nested_idx(ev, 'candidates', 0) or {}
    finish_reason = norm_finish(ev)
    parts = nested_idx(cand, 'content', 'parts') or []
    thinking = "".join(p.get("text","") for p in parts if p.get("thought") and "text" in p)
    txt = "".join(p.get("text","") for p in parts if not p.get("thought") and "text" in p)
    cites,call = grounding(cand) if 'groundingMetadata' in cand else ([], None)  # a search is reported once its grounding arrives
    tcs = _fn_calls(parts) + ([call] if call else [])
    if ev.get("error"): raise api_error_from_event(ev)
    return Delta(text=txt, thinking=thinking, tool_calls=tcs, citations=cites, finish_reason=finish_reason, usage=norm_usage(ev), raw=ev, **kwargs)

`PartAccum` asks `delta_index_fn` for the current part index and the index state to keep for the next call.

The callback accepts the current `Delta`, its part type, and the previous type and index. This implementation only needs the previous state.

These chunks contain complete calls rather than Anthropic-style indexed argument fragments. We allocate an increasing index for each collector callback. `PartAccum` handles the text and thinking accumulation.

In [45]:
#| export
def delta_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if not (last_typ or last_idx): return 0,0
    return last_idx + 1, last_idx + 1

`collate_raw` rebuilds model content for replay. It joins adjacent text parts with the same `thought` flag until a part carries `thoughtSignature`. Later text starts a new part. Non-text parts keep their order and fields.

In [46]:
#| export
def collate_raw(deltas):
    "The model content rebuilt from a stream's chunks"
    parts = []
    for ev in L(deltas).attrgot('raw'):
        for p in nested_idx(ev, 'candidates', 0, 'content', 'parts') or []:
            last = parts[-1] if parts else None
            if 'text' in p and last and 'text' in last and last.get('thought') == p.get('thought') and 'thoughtSignature' not in last:
                last['text'] += p['text']
                if 'thoughtSignature' in p: last['thoughtSignature'] = p['thoughtSignature']
            else: parts.append(dict(p))
    return dict(role='model', parts=parts)

The Gemini `acollect_stream` wrapper connects event normalization to the shared collector.

In [47]:
#| export
@delegates(mk_acollect_stream, but=['index_fn', 'api_name'])
async def acollect_stream(resp, **kwargs):
    res = mk_acollect_stream(norm_and_yield(resp, norm_sse_event), index_fn=delta_index_fn, api_name='gemini', **kwargs)
    async for o in res: yield o

In [48]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    acollect_stream=acollect_stream, raw_msg=raw_msg, collate_raw=collate_raw)
api_registry.register('gemini', **norm_funcs)

In [49]:
api_registry['gemini']

namespace(finalize_usage=<function fastcore.imports.noop(x=None, *args, **kwargs)>,
          raw_msg=<function __main__.raw_msg(resp)>,
          collate_raw=<function __main__.collate_raw(deltas)>,
          norm_tool_calls=<function __main__.norm_tool_calls(resp)>,
          norm_parts=<function __main__.norm_parts(resp)>,
          norm_finish=<function __main__.norm_finish(resp, tcs=None)>,
          norm_usage=<function __main__.norm_usage(resp)>,
          acollect_stream=<function __main__.acollect_stream(resp, *, model=None, vendor_name=None, stop_callables=None)>)

Let's compare streamed and non-streamed text, thinking, tools, and search.

#### Text

In [50]:
inp = [{"role": "user", "parts": [{"text": "Hi how are you?"}]}]
resp = await gem_call(model=mn, contents=inp)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?

::: details

- model: `models/gemini-3.8-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=6, completion_tokens=25, total_tokens=233, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=202, search_queries=0, raw={'promptTokenCount': 6, 'candidatesTokenCount': 25, 'totalTokenCount': 233, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 202, 'serviceTier': 'standard'})`

:::

In [51]:
resp = await gem_call(model=mn, contents=inp, stream=True)
async for o in acollect_stream(resp): print_stream(o)

Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?

In [52]:
o

Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=6, completion_tokens=25, total_tokens=223, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=192, search_queries=0, raw={'promptTokenCount': 6, 'candidatesTokenCount': 25, 'totalTokenCount': 223, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 192, 'serviceTier': 'standard'})`

:::

In [53]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=6, completion_tokens=25, total_tokens=233, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=202, search_queries=0, raw={'promptTokenCount': 6, 'candidatesTokenCount': 25, 'totalTokenCount': 233, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 202, 'serviceTier': 'standard'}),
 Usage(prompt_tokens=6, completion_tokens=25, total_tokens=223, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=192, search_queries=0, raw={'promptTokenCount': 6, 'candidatesTokenCount': 25, 'totalTokenCount': 223, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 192, 'serviceTier': 'standard'}))

#### Thinking

Reasoning token usage and returned thought summaries are different things. `thoughtsTokenCount` measures thinking tokens. To request summary text, set `generationConfig.thinkingConfig.includeThoughts=True`; summary parts have `thought=True`. The saved example below uses the snake-case aliases.

Signatures can arrive without summary text. Keep them for later turns rather than treating them as display text. Dropping signatures can lose reasoning context, even outside the function-calling validation described above.

Thinking defaults and controls depend on the model. For Gemini 3 models, `denorm_reasoning` sends a `thinkingLevel` and maps `minimal` to `low`. Older models take a `thinkingBudget`. Not every 2.5 model always thinks. For example, 2.5 Flash supports a zero budget. See the [generateContent thinking guide](https://ai.google.dev/gemini-api/docs/generate-content/thinking).

In [54]:
inp = [{"role": "user", "parts": [{"text": "What is 31231231 * 12312?"}]}]
eff = {"thinking_config": {"include_thoughts": True}}
resp = await gem_call(model=mn,contents=inp,generationConfig=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp


::: details
## Thinking

**My Calculation of 31,231,231 Multiplied by 12,312**

Okay, let's tackle this multiplication: 31,231,231 times 12,312. My initial thought is to break down 12,312. I could go for 12,000 + 312, or just use standard multiplication. I'll start by breaking it down.

Let A = 31,231,231.

*   A * 2 is straightforward: 62,462,462.
*   A * 10 is just a shift: 312,312,310.
*   A * 300: 9,369,369,300
*   A * 2000: 62,462,462,000
*   A * 10000: 312,312,310,000

Now, carefully sum these:

*   312,312,310,000
*   62,462,462,000 = 374,774,772,000
*   +9,369,369,300 = 384,144,141,300
*   +312,312,310 = 384,456,453,610
*   +62,462,462 = 384,518,916,072.

I need to double-check this, standard multiplication it is:

31,231,231 * 12

I observe this, but it's not truly periodic.

*   31,231,231 * 12 = 374,774,772
*   31,231,231 * 300 = 9,369,369,300
*   31,231,231 * 12 = 374,774,772
Sum: 9,744,144,072

12,312 = 12 * 1000 + 312
So:
*   374,774,772 * 1000 = 374,774,772,000
*   +9,744,144,072 = 384,518,916,072

Now, I'll check my addition:
*   374,774,772,000
*   + 9,744,144,072 = 384,518,916,072.

Finally, modular arithmetic: I have to confirm.

Modulo 9:
*   31,231,231 sums to 7, and 12,312 sums to 0. Product mod 9 should be 0.
*   384,518,916,072 sums to 54, which sums to 9, which is 0 mod 9. Matches.

Modulo 11:
*   31,231,231 alternating sum is -2 mod 11, equivalent to 9.
*   12,312 alternating sum is 3. Product is 27 = 5 mod 11.
*   384,518,916,072 alternating sum is 24 - 30 = -6, equivalent to 5 mod 11. Matches.

Last 3 digits:
*   231 * 312 = 72,072, so the last three digits should be 072, and they are.

I am confident in this final answer: 384,518,916,072.




:::

To find the product of $31,231,231$ and $12,312$, we can break down the multiplication:

$$12,312 = 12,000 + 312$$

1. **Multiply by $12$:**
   $$31,231,231 \times 12 = 374,774,772$$

   So, 
   $$31,231,231 \times 12,000 = 374,774,772,000$$

2. **Multiply by $312$:**
   $$31,231,231 \times 300 = 9,369,369,300$$
   $$31,231,231 \times 12 = 374,774,772$$
   $$9,369,369,300 + 374,774,772 = 9,744,144,072$$

3. **Add the two parts together:**
   $$374,774,772,000 + 9,744,144,072 = 384,518,916,072$$

**Answer:**
**384,518,916,072**

::: details

- model: `models/gemini-3.8-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=20, completion_tokens=350, total_tokens=1774, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=1404, search_queries=0, raw={'promptTokenCount': 20, 'candidatesTokenCount': 350, 'totalTokenCount': 1774, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 1404, 'serviceTier': 'standard'})`

:::

In [55]:
resp = await gem_call(model=mn, contents=inp, generationConfig=eff, stream=True)
async for o in acollect_stream(resp): print_stream(o)

🧠🧠To calculate **31,231,231 × 12,312**, we can break it down step-by-step:

$$\begin{aligned}
31{,}231{,}231 \times 2 &= 62{,}462{,}462 \\
31{,}231{,}231 \times 10 &= 312{,}312{,}310 \\
31{,}231{,}231 \times 300 &= 9{,}369{,}369{,}300 \\
31{,}231{,}231 \times 2{,}000 &= 62{,}462{,}462{,}000 \\
31{,}231{,}231 \times 10{,}000 &= 312{,}312{,}310{,}000
\end{aligned}$$

Now, add these partial products together:

$$\begin{array}{r@{\quad}l}
& 312{,}312{,}310{,}000 \\
+ & 62{,}462{,}462{,}000 \\
+ & 9{,}369{,}369{,}300 \\
+ & 312{,}312{,}310 \\
+ & 62{,}462{,}462 \\
\hline
& \mathbf{384{,}518{,}916{,}072}
\end{array}$$

**Answer:**
**384,518,916,072**

In [56]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [57]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=20, completion_tokens=350, total_tokens=1774, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=1404, search_queries=0, raw={'promptTokenCount': 20, 'candidatesTokenCount': 350, 'totalTokenCount': 1774, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 1404, 'serviceTier': 'standard'}),
 Usage(prompt_tokens=20, completion_tokens=370, total_tokens=1907, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=1517, search_queries=0, raw={'promptTokenCount': 20, 'candidatesTokenCount': 370, 'totalTokenCount': 1907, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 1517, 'serviceTier': 'standard'}))

#### Tool call

In [58]:
inp = [{"role": "user", "parts": [{"text": 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]
tools = [{"functionDeclarations": [sch['function']]}]
resp = await gem_call(model=mn, contents=inp, tools=tools)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp



🔧 simple_add({'b': 5, 'a': 3})


🔧 simple_add({'a': 10, 'b': 20})


::: details

- model: `models/gemini-3.8-flash`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=77, completion_tokens=38, total_tokens=180, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=65, search_queries=0, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 180, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 65, 'serviceTier': 'standard'})`

:::

In [59]:
resp = await gem_call(model=mn, contents=inp, tools=tools, stream=True)
async for o in acollect_stream(resp): print_stream(o)


- ⏳ `simple_add(b="5", a="3")` ⏳

- ⏳ `simple_add(b="20", a="10")` ⏳


In [60]:
def test_tcs(tcs1,tcs2):
    for tc1,tc2 in zip(tcs1,tcs2): 
        test_eq(tc1.name,tc2.name)
        test_eq(tc1.arguments,tc2.arguments)
        test_eq(tc1.server,tc2.server)

In [61]:
test_tcs(comp.tool_calls,o.tool_calls)

In [62]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].raw.keys()), set(o.message.content[0].raw.keys()))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=77, completion_tokens=38, total_tokens=180, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=65, search_queries=0, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 180, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 65, 'serviceTier': 'standard'}),
 Usage(prompt_tokens=77, completion_tokens=38, total_tokens=199, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=84, search_queries=0, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 199, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 84, 'serviceTier': 'standard'}))

#### Web search

Search runs on the server. In this search-only example, grounding metadata describes the work without a client `functionCall`. `norm_parts` puts the synthesized `google_search` call before the text, then attaches citations to the first text part. `norm_usage` records the query count.


In [63]:
inp = [{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}]
resp = await gem_call(model=mn, contents=inp, tools=[{"googleSearch": {}}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
test_eq(comp.tool_calls[0].name, 'google_search')
assert 'Search suggestion: [' in comp.tool_calls[0].text, comp.tool_calls[0]
comp.tool_calls[0]

🔧 **google_search**(`{'queries': ['Istanbul weather current conditions celsius', 'weather in Istanbul today']}`)

::: details

- id: `google_search`
- server: `True`
- raw: `None`

:::

In [64]:
resp = await gem_call(model=mn, contents=inp, tools=[{"googleSearch": {}}], stream=True)
async for o in acollect_stream(resp): pass
o

The weather in Istanbul is predominantly **overcast and breezy with rain showers**:

* **Current Conditions:** Around **16°C – 17°C** (61°F – 62°F) under cloudy skies with light rain showers.
* **Temperatures:** Expected daytime high around **19°C – 20°C** (66°F – 68°F), with nighttime lows near **16°C – 17°C** (61°F – 62°F).
* **Precipitation:** Intermittent rain and passing showers throughout the day, with periods of steadier rain.
* **Wind:** Moderate to fresh north/northeast winds around **20–30 km/h** (12–18 mph), with occasional gusts up to **50 km/h** (30 mph).
* **Humidity:** High, hovering around **75% – 85%**.

If you are heading out, bringing an umbrella or a rain jacket is recommended.

🔧 google_search({'queries': ['current weather in Istanbul Turkey', 'Istanbul weather forecast today']})


::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=65, completion_tokens=335, total_tokens=806, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=406, search_queries=2, raw={'promptTokenCount': 65, 'candidatesTokenCount': 335, 'totalTokenCount': 806, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 65}], 'thoughtsTokenCount': 406, 'serviceTier': 'standard', 'server_tool_use': {'google_search': 2}})`

:::

In the recorded stream, grounding arrives at the end. `acollect_stream` yields its citations then, too late to annotate text as it first appears. Anthropic's citation deltas can arrive alongside text instead.

In [65]:
test_eq(comp.finish_reason, o.finish_reason)

In [66]:
comp.usage, o.usage

(Usage(prompt_tokens=134, completion_tokens=317, total_tokens=727, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=276, search_queries=2, raw={'promptTokenCount': 134, 'candidatesTokenCount': 317, 'totalTokenCount': 727, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 134}], 'thoughtsTokenCount': 276, 'serviceTier': 'standard', 'server_tool_use': {'google_search': 2}}),
 Usage(prompt_tokens=65, completion_tokens=335, total_tokens=806, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=406, search_queries=2, raw={'promptTokenCount': 65, 'candidatesTokenCount': 335, 'totalTokenCount': 806, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 65}], 'thoughtsTokenCount': 406, 'serviceTier': 'standard', 'server_tool_use': {'google_search': 2}}))

When function declarations accompany a supported server tool, `mk_payload` requests `includeServerSideToolInvocations`. The search example then returns signed `toolCall` and `toolResponse` parts. Its response part contains the suggestion widget.

The canonical message has one search call from `grounding`, not duplicate calls for those blocks. `Msg.raw` retains both original blocks and their signatures for replay.


In [67]:
srch_prompt = 'Search the web for the current stable Python 3 minor version number, then call simple_add to add that number to 100. One short sentence.'
srch_tools = [dict(functionDeclarations=[sch['function']]), {"googleSearch": {}}]
resp = await gem_call(model=mn, contents=[{"role": "user", "parts": [{"text": srch_prompt}]}], tools=srch_tools, toolConfig=dict(includeServerSideToolInvocations=True))
comp = mk_completion(resp, mn, api_name, vnd_nm)
test_eq([(t.name, t.server) for t in comp.tool_calls], [('google_search', True), ('simple_add', False)])
assert comp.usage.raw['server_tool_use']['google_search'] >= 1
[next(k for k in p if k != 'thoughtSignature') for p in comp.message.raw['parts']]

['toolCall', 'toolResponse', 'functionCall']

## Denormalize

The request converters turn shared messages and options into Gemini fields.

### Msgs

In [68]:
def mk_user_msg(txt): return Msg(role='user', content=[Text(txt)])

Assistant messages use Gemini's `model` role. User messages and tool replies use `user`; each tool reply contains a `functionResponse`.

This first tool-result converter handles text. The exported version below also handles media.

In [69]:
def denorm_tool_result(p:ToolResult):
    "Convert canonical tool_result Part to Gemini functionResponse part."
    fr = dict(name=p.name or '', response={"content": tool_text(p.text)})
    if p.id: fr['id'] = p.id
    return dict(functionResponse=fr)

This first user-message converter handles text. The exported version below adds media inputs.

In [70]:
def denorm_user(m:Msg):
    "Convert canonical user Msg to Gemini user content."
    parts = [dict(text=p.text or '') for p in m.content if isinstance(p, Text)]
    return dict(role='user', parts=parts)

In [71]:
#| export
def denorm_tool_use(p:ToolUse):
    "Convert canonical tool_use Part to Gemini functionCall part."
    fc = dict(name=p.name or '', args=p.arguments or {})
    if p.id: fc['id'] = p.id
    return dict(functionCall=fc)

def denorm_assistant(m:Msg):
    "The content as the API sent it, else its text and tool calls"
    if m.raw: return m.raw
    parts = [dict(text=p.text or '') if isinstance(p, Text) else denorm_tool_use(p) for p in m.content if isinstance(p, (Text, ToolUse))]
    return dict(role='model', parts=parts)

def denorm_tool(m:Msg):
    "Convert canonical tool Msg to Gemini user content with functionResponse parts."
    parts = [denorm_tool_result(p) for p in m.content if isinstance(p, ToolResult)]
    return dict(role='user', parts=parts)

def denorm_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to Gemini contents."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_user(m))
        elif m.role == 'assistant': res.append(denorm_assistant(m))
        elif m.role == 'tool':      res.append(denorm_tool(m))
    return res

`denorm_assistant` reuses `Msg.raw` when present. In this tool round, the search call and response keep their signatures. We add only the client function's result.

In [72]:
tres = Msg('tool', [ToolResult(id=comp.tool_calls[-1].id, name='simple_add', text='114')])
turn = [Msg('user', [Text(srch_prompt)]), comp.message, tres]
wire = denorm_msgs(turn)
test_is(wire[1], comp.message.raw)
resp = await gem_call(model=mn, contents=wire, tools=srch_tools, toolConfig=dict(includeServerSideToolInvocations=True))
mk_completion(resp, mn, api_name, vnd_nm).message.text

'The current stable Python 3 minor version is 14, and adding it to 100 results in 114.'

The `hist2fmt`/`fmt2hist` round trip drops `raw`. The converter then rebuilds text and ordinary function calls without signatures. The stored search becomes a function-call/result pair.

The new user turn asks to add 1 to the previous result. Gemini returns another `simple_add` call, with arguments 114 and 1. We display the call rather than `.message.text`, which is empty while the model waits for the tool result.

In [73]:
stored = fmt2hist(hist2fmt(turn[1:]))
wire = denorm_msgs([turn[0], *stored, Msg('user', [Text('Now add 1 to that. One short sentence.')])])
test_eq([list(p)[0] for p in wire[1]['parts']][-2:], ['functionCall', 'functionCall'])
assert not any('thoughtSignature' in p for p in wire[1]['parts'])
test_eq([list(p)[0] for p in wire[2]['parts']], ['functionResponse', 'functionResponse'])
resp = await gem_call(model=mn, contents=wire, tools=srch_tools, toolConfig=dict(includeServerSideToolInvocations=True))
[(tc.name, tc.arguments) for tc in mk_completion(resp, mn, api_name, vnd_nm).tool_calls]

[('simple_add', {'b': 1, 'a': 114})]

### Tool Schemas

Gemini's [function declarations](https://ai.google.dev/gemini-api/docs/function-calling#function-declarations) accept a subset of OpenAPI schema. `denorm_tool_schs` groups functions under `functionDeclarations` and retains other tools separately. It filters schema keywords at every level, including nested properties and array items. Property names, defaults, and example values stay intact.

The tuple example below shows how the converter removes unsupported `prefixItems`.

In [74]:
#| export
_valid_gemini_sch = {'type', 'format', 'title', 'description', 'nullable', 'default', 'items', 'minItems', 'maxItems', 'enum', 'properties',
    'propertyOrdering', 'required', 'minProperties', 'maxProperties', 'minimum', 'maximum', 'minLength', 'maxLength', 'pattern', 'example',
    'anyOf'}

def _gem_filter_sch(s):
    if isinstance(s, list): return [_gem_filter_sch(x) for x in s]
    if not isinstance(s, dict): return s
    res = {}
    for k,v in s.items():
        if k not in _valid_gemini_sch: continue
        if k == 'properties': res[k] = {name:_gem_filter_sch(prop) for name,prop in v.items()}
        elif k in ('items', 'anyOf'): res[k] = _gem_filter_sch(v)
        else: res[k] = v
    return res

def denorm_tool_schs(tools):
    "Convert canonical tools to Gemini format."
    fn_decls, other = [], []
    for t in tools:
        fn = fn_schema(t)
        if fn is None: other.append(t); continue
        name, desc, params = fn
        params = _gem_filter_sch(params)
        fn_decls.append(dict(name=name, description=desc, parameters=params))
    out = other[:]
    if fn_decls: out.insert(0, dict(functionDeclarations=fn_decls))
    return out

In [75]:
def view_test(
    path:str, # Path to directory or file to view
    view_range:tuple[int,int]=None, # Optional 1-indexed (start, end) line range for files, end=-1 for EOF. Do NOT use unless it's known that the file is too big to keep in context—simply view the WHOLE file when possible
    nums:bool=False, # Whether to show line numbers
    skip_folders:tuple[str,...]=('_proc','__pycache__') # Folder names to skip when listing directories
):
    'Test tool'
    pass

In [76]:
invalid_sch = lite_mk_func(view_test)
gem_func = denorm_tool_schs([invalid_sch])
test_eq('prefixItems' not in gem_func[0]['functionDeclarations'][0]['parameters']['properties']['view_range'], True)

In [77]:
test_eq(denorm_tool_schs([sch]), [{'functionDeclarations': [dict(name='simple_add', description='add numbers', parameters=sch['function']['parameters'])]}])
test_eq(denorm_tool_schs([{"googleSearch": {}}]), [{"googleSearch": {}}])
test_eq(denorm_tool_schs([sch, {"googleSearch": {}}]), [{'functionDeclarations': [dict(name='simple_add', description='add numbers', parameters=sch['function']['parameters'])]}, {"googleSearch": {}}])

### Tool Choice

In [78]:
#| export
def denorm_tool_choice(v):
    "Map canonical tool_choice to Gemini toolConfig."
    if v is None: return None
    if v in ('auto',):                    return {'functionCallingConfig': {'mode': 'AUTO'}}
    if v in ('required', 'any', 'force'): return {'functionCallingConfig': {'mode': 'ANY'}}
    if v in ('none', 'off', 'disabled'):  return {'functionCallingConfig': {'mode': 'NONE'}}
    return {'functionCallingConfig': {'mode': 'ANY', 'allowedFunctionNames': [v]}}

Modes

In [79]:
test_eq(denorm_tool_choice('auto'), {'functionCallingConfig': {'mode': 'AUTO'}})
test_eq(denorm_tool_choice('none'), {'functionCallingConfig': {'mode': 'NONE'}})

Tool name

In [80]:
test_eq(denorm_tool_choice('simple_add'), {'functionCallingConfig': {'mode': 'ANY', 'allowedFunctionNames': ['simple_add']}})

None

In [81]:
test_eq(denorm_tool_choice(None), None)

### Reasoning Effort

For model names containing `gemini-3`, string efforts set `thinkingLevel`: `minimal` maps to `low`, and `low`, `medium`, and `high` retain their names. Other model names use budgets of 128, 1024, 2048, and 4096 respectively. Both paths request summaries with `includeThoughts=True`.

A dict passes through as the thinking configuration. `None` leaves it unset. Use one of the supported strings or a dict for explicit provider settings.

Other inputs raise `ValueError`.

In [82]:
#| export
_gem_think_budgets = dict(minimal=128, low=1024, medium=2048, high=4096)
_gem_think_levels  = dict(minimal='low', low='low', medium='medium', high='high')

def denorm_reasoning(v, model=''):
    "Map canonical reasoning_effort to Gemini thinkingConfig (uses thinkingLevel for Gemini 3+)."
    if v is None or isinstance(v, dict): return v
    if not isinstance(v, str) or v not in _gem_think_budgets:
        raise ValueError(f"Invalid reasoning effort for Gemini: {v!r}; expected one of {list(_gem_think_budgets)}, a dict, or None")
    if 'gemini-3' in model: return {'thinkingLevel': _gem_think_levels[v], 'includeThoughts': True}
    return {'thinkingBudget': _gem_think_budgets[v], 'includeThoughts': True}

In [83]:
test_eq(denorm_reasoning('medium', 'models/gemini-2.5-flash'), {'thinkingBudget': 2048, 'includeThoughts': True})
test_eq(denorm_reasoning('low', flash), {'thinkingLevel': 'low', 'includeThoughts': True})
test_eq(denorm_reasoning('high', flash), {'thinkingLevel': 'high', 'includeThoughts': True})
test_eq(denorm_reasoning(None), None)

### Web Search Options

Any supplied options map to `{'googleSearch': {}}`. This converter does not interpret context size or location.

In [84]:
#| export
def denorm_web_search(v): return {"googleSearch": {}}

### System

In [85]:
#| export
def denorm_system(sp): return dict(parts=[{'text': sys_text(part_txt(sp))}])

In [86]:
sp = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
msg1 = mk_user_msg('What are you?')

`denorm_system` wraps the prompt text in `systemInstruction.parts`, outside the conversation contents.

In [87]:
resp = await gem_call(model=mn, contents=denorm_msgs([msg1]), systemInstruction=denorm_system(sp), stream=True)
async for o in acollect_stream(resp): pass
o

Arrr, I be Gemini 3.8 Flash, a mighty large language model forged by the clever crew at Google, sailin' the digital seas to do yer biddin'!

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=22, completion_tokens=38, total_tokens=465, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=405, search_queries=0, raw={'promptTokenCount': 22, 'candidatesTokenCount': 38, 'totalTokenCount': 465, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 22}], 'thoughtsTokenCount': 405, 'serviceTier': 'standard'})`

:::

### Media Inputs

`InputImage`, `InputAudio`, `InputVideo`, and `InputFile` are shared `Media` parts. Their `text` holds a URL or data URL, and `mime` records the type.

These Gemini converters derive the request's MIME type from the data URL or `url_mime`, rather than reading `p.mime`. Data URLs become `inlineData` with base64 data. Remote URLs become `fileData` with a `fileUri`.

In [88]:
#| export
def denorm_user(m:Msg):
    "Convert canonical user Msg to Gemini user content."
    parts = []
    for p in m.content:
        if   isinstance(p, Text):       parts.append({"text": p.text or ""})
        elif isinstance(p, InputImage): parts.append(denorm_image(p))
        elif isinstance(p, InputAudio): parts.append(denorm_audio(p))
        elif isinstance(p, InputVideo): parts.append(denorm_video(p))
        elif isinstance(p, InputFile):  parts.append(denorm_file(p))
    return dict(role='user', parts=parts)

#### Image

In [89]:
#| export
def denorm_image(p):
    if (b64:=data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": url_mime(p.text, "image/*"), "fileUri": p.text}}

Image URL

In [90]:
msg1 = Msg(role='user', content=[InputImage(sample_img_url), Text('What is this image?')])

In [91]:
resp = await gem_call(model=mn, contents=denorm_msgs([msg1]),stream=True)
async for o in acollect_stream(resp): pass
o

This is a scenic landscape image—commonly used as a stock background, wallpaper, or product display mockup—composed of:

* **Foreground:** A rustic wooden dock or plank tabletop viewed in perspective, creating a viewing platform.
* **Middle ground:** A calm, glassy alpine lake (resembling famous glacial lakes like **Lake Matheson** in New Zealand) reflecting the surrounding lush green forest and mountains.
* **Background:** A majestic, snow-capped mountain range under a bright, slightly hazy blue sky.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=1086, completion_tokens=104, total_tokens=1336, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=146, search_queries=0, raw={'promptTokenCount': 1086, 'candidatesTokenCount': 104, 'totalTokenCount': 1336, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}, {'modality': 'IMAGE', 'tokenCount': 1080}], 'thoughtsTokenCount': 146, 'serviceTier': 'standard'})`

:::

Image data

In [92]:
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="
msg1 = Msg(role='user', content=[InputImage(img_b64), Text('What color is this pixel?')])

In [93]:
resp = await gem_call(model=mn, contents=denorm_msgs([msg1]),stream=True)
async for o in acollect_stream(resp): pass
o

That pixel is **red** (specifically pure red, hex code `#FF0000` / RGB `(255, 0, 0)`).

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=1096, completion_tokens=35, total_tokens=1172, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=41, search_queries=0, raw={'promptTokenCount': 1096, 'candidatesTokenCount': 35, 'totalTokenCount': 1172, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 7}, {'modality': 'IMAGE', 'tokenCount': 1089}], 'thoughtsTokenCount': 41, 'serviceTier': 'standard'})`

:::

#### Audio

In [94]:
#| export
def denorm_audio(p): 
    if (b64:=data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": url_mime(p.text, "audio/*"), "fileUri": p.text}}

In [95]:
wav_data = httpx2.get("https://openaiassets.blob.core.windows.net/$web/API/docs/audio/alloy.wav").content
audio_b64 = f"data:audio/wav;base64,{base64.b64encode(wav_data).decode()}"

In [96]:
msg1 = Msg(role='user', content=[InputAudio(audio_b64), Text('What is this audio saying?')])

In [97]:
resp = await gem_call(model=mn, contents=denorm_msgs([msg1]),stream=True)
async for o in acollect_stream(resp): pass
o

The audio says:  "The sun rises in the east and sets in the west. This simple fact has been observed by humans for thousands of years."

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=181, completion_tokens=31, total_tokens=261, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=49, search_queries=0, raw={'promptTokenCount': 181, 'candidatesTokenCount': 31, 'totalTokenCount': 261, 'promptTokensDetails': [{'modality': 'AUDIO', 'tokenCount': 174}, {'modality': 'TEXT', 'tokenCount': 7}], 'thoughtsTokenCount': 49, 'serviceTier': 'standard'})`

:::

#### Video

In [98]:
#| export
def denorm_video(p):
    if (b64:=data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": url_mime(p.text, "video/mp4"), "fileUri": p.text}}

The example sends a YouTube URL. Gemini 3.8 Flash rejects other external video URLs, such as a public Cloud Storage MP4, with `INVALID_ARGUMENT`. Send other videos as data URLs, which Gemini's converter also handles. Video isn't exclusive to this backend. The local Responses converter emits `input_video`, although actual acceptance depends on the selected API and model.

In [99]:
vid_yt = "https://www.youtube.com/watch?v=9hE5-98ZeCg"
msg1 = Msg(role='user', content=[InputVideo(vid_yt), Text('Summarize this video in one sentence.')])

In [100]:
resp = await gem_call(model=mn, contents=denorm_msgs([msg1]),stream=True)
async for o in acollect_stream(resp): pass
o

A user demonstrates Gemini 2.0's Multimodal Live API by showcasing real-time screen sharing, reading and defining text, handling interruptions, and summarizing the interaction.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=13576, completion_tokens=35, total_tokens=13786, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=175, search_queries=0, raw={'promptTokenCount': 13576, 'candidatesTokenCount': 35, 'totalTokenCount': 13786, 'promptTokensDetails': [{'modality': 'VIDEO', 'tokenCount': 13567}, {'modality': 'TEXT', 'tokenCount': 9}], 'thoughtsTokenCount': 175, 'serviceTier': 'standard'})`

:::

#### Files

`denorm_file` uses `mimeType` for both inline and remote files. It takes the type from the data URL, or uses `url_mime` with an `application/pdf` fallback. The OpenAI data-URL file converters add a filename; Anthropic's base64 document source calls the type field `media_type`.

In [101]:
#| export
def denorm_file(p):
    if (b64:=data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": url_mime(p.text, "application/pdf"), "fileUri": p.text}}

In [102]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
msg1 = Msg(role='user', content=[InputFile(pdf_url), Text('What is the title of this paper?')])

In [103]:
resp = await gem_call(model=mn, contents=denorm_msgs([msg1]),stream=True)
async for o in acollect_stream(resp): pass
o

The title of the paper is **"Attention Is All You Need"**.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=7809, completion_tokens=15, total_tokens=7853, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=29, search_queries=0, raw={'promptTokenCount': 7809, 'candidatesTokenCount': 15, 'totalTokenCount': 7853, 'promptTokensDetails': [{'modality': 'IMAGE', 'tokenCount': 7800}, {'modality': 'TEXT', 'tokenCount': 9}], 'thoughtsTokenCount': 29, 'serviceTier': 'standard'})`

:::

In [104]:
pdf_b64 = f"data:application/pdf;base64,{base64.b64encode(httpx2.get(pdf_url).content).decode()}"
msg1 = Msg(role='user', content=[InputFile(pdf_b64), Text('What is the title of this paper')])

In [105]:
resp = await gem_call(model=mn, contents=denorm_msgs([msg1]),stream=True)
async for o in acollect_stream(resp): pass
o

The title of the paper is **Attention Is All You Need**.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=7808, completion_tokens=13, total_tokens=7850, cached_tokens=6663, cache_creation_tokens=0, reasoning_tokens=29, search_queries=0, raw={'promptTokenCount': 7808, 'candidatesTokenCount': 13, 'totalTokenCount': 7850, 'cachedContentTokenCount': 6663, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 8}, {'modality': 'IMAGE', 'tokenCount': 7800}], 'cacheTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}, {'modality': 'IMAGE', 'tokenCount': 6657}], 'thoughtsTokenCount': 29, 'serviceTier': 'standard'})`

:::

### Media Tool Call Results

A list-valued tool result can contain `Text`, `InputImage`, and `InputFile` parts. These go inside `functionResponse.parts`, with empty `response.content`. Other part types raise `ValueError`. Non-list results use `tool_text` in `response.content`. The converter includes the function name and, when present, its ID.

In [106]:
#| export
def denorm_tool_result(p:ToolResult):
    "Convert canonical tool_result Part to Gemini functionResponse part."
    fr = dict(name=p.name or '', response={"content": "" if isinstance(p.text, list) else tool_text(p.text)})
    if p.id: fr['id'] = p.id
    if isinstance(p.text, list):
        parts = []
        for pp in p.text:
            if   isinstance(pp, Text):       parts.append({"text": pp.text or ""})
            elif isinstance(pp, InputImage): parts.append(denorm_image(pp))
            elif isinstance(pp, InputFile):  parts.append(denorm_file(pp))
            else: raise ValueError(f"Gemini tool_result does not support {pp.type}")
        fr['parts'] = parts
    return dict(functionResponse=fr)

In [107]:
def test_img(): "Show the test image"
img_tools = denorm_tool_schs([lite_mk_func(test_img)])
img_q = mk_user_msg('Call test_img, then describe what it shows in one sentence.')
resp = await gem_call(model=mn, contents=denorm_msgs([img_q]), tools=img_tools)
comp = mk_completion(resp, mn, api_name, vnd_nm)
msgs = [img_q, comp.message, Msg(role='tool', content=[ToolResult(id=comp.tool_calls[0].id, name='test_img', text=[InputImage(img_b64)])])]
comp.tool_calls


[ToolUse(raw={'thoughtSignature': 'Et0BCtoBAWkUfRPQa7wNS6dsuenTYPbEi+iEJB0T/pSaedvLQaZ+le+Wq87mO4srESGkiIZb43UMUTUaUvDgnyHfBqIxGVeph9IWycmiUqwwr7G8tAhS+ebBakIPDQSLFrEDGNZumAyoOCIqgy+udCORbjCCLzs4VA1CfxVlQWZmt8Kat9xoETwgCNMBwuU2qdqxKCyAWFtEdxFesVRXcAWBHJSgJL63HuMW7JhMUHIPvMhXQAeN/x2IkWa55Q2T5Pr6DGnNtSl+P6p8OrU+huMIqzcw4svZjo0VQ9R3/Rg='}, cache_control=None, id='call_198612', name='test_img', arguments={}, server=False, text=None)]

In [108]:
resp = await gem_call(model=mn, contents=denorm_msgs(msgs), tools=img_tools, stream=True)
async for o in acollect_stream(resp): pass
o


The image displays a solid, uniform square of bright red color.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=1182, completion_tokens=13, total_tokens=1215, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=20, search_queries=0, raw={'promptTokenCount': 1182, 'candidatesTokenCount': 13, 'totalTokenCount': 1215, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 93}, {'modality': 'IMAGE', 'tokenCount': 1089}], 'thoughtsTokenCount': 20, 'serviceTier': 'standard'})`

:::

## Payload

`mk_payload` builds `contents`, `systemInstruction`, tools, and `generationConfig`. It maps token limits to `maxOutputTokens` and preserves a zero temperature. Function declarations combined with Google Search, code execution, or Google Search Retrieval enable `includeServerSideToolInvocations`. Tool-choice fields share that `toolConfig`.

In [109]:
#| export
@delegates(payload_kwargs)
def mk_payload(msgs, model, **kwargs):
    payload = dict(contents=denorm_msgs(msgs))
    if sp:=kwargs.get('system'): payload['systemInstruction'] = denorm_system(sp)
    gen_config = {}
    if mt:=kwargs.get('max_tokens'):        gen_config['maxOutputTokens'] = mt
    if thk:=denorm_reasoning(kwargs.get('reasoning_effort'), model): gen_config['thinkingConfig'] = thk
    if (temp:=kwargs.get('temperature')) is not None: gen_config['temperature'] = temp
    if gen_config: payload['generationConfig'] = gen_config
    gem_tools = denorm_tool_schs(kwargs.get('tools')) if kwargs.get('tools') else []
    if (wopts:=kwargs.get('web_search_options')) is not None: gem_tools.append(denorm_web_search(wopts))
    if gem_tools:
        payload['tools'] = gem_tools
        has_fn  = any('functionDeclarations' in t for t in gem_tools)
        has_srv = any(k in t for t in gem_tools for k in ('googleSearch','codeExecution','googleSearchRetrieval'))
        if has_fn and has_srv: payload.setdefault('toolConfig', {})['includeServerSideToolInvocations'] = True
    if tchc:=denorm_tool_choice(kwargs.get('tool_choice')): payload.setdefault('toolConfig', {}).update(tchc)
    return payload

`get_hdrs` uses bearer authentication when given `oauth_token`. Otherwise it sends `x-goog-api-key` from the explicit key or `GEMINI_API_KEY`.

In [110]:
#| export
def get_hdrs(api_key=None, oauth_token=None):
    "Bearer auth for an OAuth token; otherwise the API key header"
    if oauth_token: return {"Authorization": f"Bearer {oauth_token}"}
    return {"x-goog-api-key": get_api_key(api_key, 'GEMINI_API_KEY')}


## Cost

`cost` selects a price tier from prompt length and model metadata. It charges uncached input, candidate output, cache reads, and thinking tokens separately. A dedicated reasoning rate takes precedence over the normal output rate. Audio input gets its own rate when metadata supplies one; otherwise it stays in the standard input count.

In [111]:
#| export
def cost(usage, m):
    raw = usage.raw
    prompt_tot = raw.get('promptTokenCount', 0)
    tier = price_tier(m, prompt_tot)
    in_rate    = tier_rate(m, 'input_cost_per_token', tier)
    out_rate   = tier_rate(m, 'output_cost_per_token', tier)
    cache_rate = tier_rate(m, 'cache_read_input_token_cost', tier)
    # Models with no `input_cost_per_audio_token` (e.g. Gemini 3 Pro) bill audio at the standard input rate, so it stays in `in_txt`
    audio_rate = m.get('input_cost_per_audio_token')
    cached = raw.get('cachedContentTokenCount', 0)
    audio  = sum(d['tokenCount'] for d in raw.get('promptTokensDetails', []) if d.get('modality')=='AUDIO') if audio_rate else 0
    in_txt = prompt_tot - cached - audio
    thoughts = raw.get('thoughtsTokenCount', 0) or 0
    cands    = raw.get('candidatesTokenCount', 0) or 0
    reason_rate = m.get('output_cost_per_reasoning_token') or out_rate
    cost  = in_txt * in_rate + cands * out_rate
    cost += cached * cache_rate
    cost += audio  * (audio_rate or 0)
    cost += thoughts * reason_rate
    return cost

## Register API

The final registry entry supplies response normalizers, the payload builder, stream collector, headers, and cost function. `raw_msg` and `collate_raw` keep replay data. `endpoint(model, stream)` supplies the request path and query parameters. `acomplete` uses this entry to send requests.

In [112]:
#| export
api_ns = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    acollect_stream=acollect_stream, raw_msg=raw_msg, collate_raw=collate_raw, mk_payload=mk_payload, cost=cost, get_hdrs=get_hdrs,
    endpoint=endpoint)
api_registry.register('gemini', **api_ns)

## Costs through `acomplete`

`acomplete` sends Gemini models through this adapter. These checks compare `Completion.cost` between streamed and non-streamed `acomplete` calls. The last check uses the pricing tier above 200,000 tokens.

Basic call:

In [113]:
comp = await acomplete([Msg('user', content=[Text("Say hello in French")])], model=flash, max_tokens=64, temperature=0.0)
comp.cost

7.125e-05

In [114]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Text("Say hello in French")])], model=flash, max_tokens=64, temperature=0.0, stream=True)
async for scomp in it: pass
test_close(scomp.cost, comp.cost, 1e-7)

Thinking:

In [115]:
comp = await acomplete([Msg('user', content=[Text('What is 123 * 456?')])], model=flash, reasoning_effort='low', temperature=0.0)
comp.cost

7.35e-05

In [116]:
# Streaming - as a sanity check; loose tolerance since thought-token counts vary between runs
it = await acomplete([Msg('user', content=[Text('What is 123 * 456?')])], model=flash, reasoning_effort='low', temperature=0.0, stream=True)
async for scomp in it: pass
test_close(scomp.cost, comp.cost, 1e-3)

Audio:

In [117]:
msg = Msg('user', content=[InputAudio(audio_b64), Text('What is this audio saying?')])
comp = await acomplete([msg], model=flash, max_tokens=64, temperature=0.0)
comp.cost

0.00024825

In [118]:
# Streaming - as a sanity check
it = await acomplete([msg], model=flash, max_tokens=64, temperature=0.0, stream=True)
async for scomp in it: pass
test_close(scomp.cost, comp.cost, 1e-4)

#### Caching

In [119]:
big_text = 'The quick brown fox jumps over the lazy dog. ' * 250
msg1 = Msg('user', content=[Text(big_text + "\n\nWrite sentence that is repeated")])
comp1 = await acomplete([msg1], model=flash, temperature=0.0)

In [120]:
msg3 = Msg('user', content=[Text('Now in French!')])
comp2 = await acomplete([msg1, comp1.message, msg3], model=flash, temperature=0.0)  # should read
print('cached:', comp2.usage.raw.get('cachedContentTokenCount', 0))

cached: 0


Gemini does not guarantee an implicit cache hit. Inspect the usage record to see whether this request read cached tokens; a repeated prompt alone is not a deterministic cache test.

In [121]:
comp2.usage.raw

{'promptTokenCount': 2773,
 'candidatesTokenCount': 64,
 'totalTokenCount': 2837,
 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 2773}],
 'serviceTier': 'standard'}

#### Large context

Gemini 3.8 Flash has no long-context pricing tier. This offline check uses Gemini 3.1 Pro's metadata, which prices prompts above 200,000 tokens at a higher rate:

In [122]:
fake_usage = Usage(prompt_tokens=250_000, completion_tokens=1000, total_tokens=251_000,
    raw=dict(promptTokenCount=250_000, candidatesTokenCount=1000, totalTokenCount=251_000))
pro_meta = dict2obj(get_model_meta('models/gemini-3.1-pro-preview', 'gemini'))
big_cost = cost(fake_usage, pro_meta)
expected = 250_000 * pro_meta.input_cost_per_token_above_200k_tokens + 1000 * pro_meta.output_cost_per_token_above_200k_tokens
test_close(big_cost, expected, 1e-10)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()